# 6.7 - Latent factors: hidden drivers of returns, against the risk model's named ones

The risk model of notebook 4.3 explains returns with factors someone *named* in advance: the market, eleven
industries and twelve styles (size, value, momentum, volatility and so on), following BlackRock's BFRE. Machine
learning offers another route: let the data find the factors. This notebook estimates such **latent factors** four
ways - PCA, RP-PCA, IPCA and a conditional autoencoder - and asks, out of sample over 2015-2024:

1. How much of the variation in returns does each explain?
2. How well does each **price** the cross-section - are average returns left unexplained?
3. How good an investment is each model's best combination of its factors (its *stochastic discount factor*)?

All four are compared on a common ground: 46 **characteristic-managed portfolios**. Each month, one portfolio per
feature of notebook 6.2, holding every S&P 1500 member in proportion to its (centred) feature, plus an equal-weighted
market portfolio:
$$x_{t+1} = \frac{1}{N_t} Z_t^\top r_{t+1},$$
with $Z_t$ the members' characteristics (one row a company) and $r_{t+1}$ their next-month returns (each date's
extreme 1 % clipped). These portfolios span what the characteristics can earn, which is what a factor model must
explain.

## The four methods

| Method | Factors | Loadings |
|---|---|---|
| **PCA** | the $K$ directions in which the managed portfolios vary most (eigenvectors of $X^\top X / T$) | fixed |
| **RP-PCA** (Lettau and Pelger, 2020) | the eigenvectors of $X^\top X / T + \gamma \bar x \bar x^\top$, $\gamma$ = 10: directions that vary *or earn* | fixed |
| **IPCA** (Kelly, Pruitt and Su, 2019) | $K$ factors, re-estimated each month by regressing returns on the loadings | each stock's loadings a linear function of its characteristics, $\beta_{i,t} = z_{i,t}^\top \Gamma$ |
| **Conditional autoencoder** (Gu, Kelly and Xiu, 2021) | $K$ linear combinations of the managed portfolios | a small neural network of the characteristics (one hidden layer of 32 units) |
| **BFRE** (the risk model, 4.3) | the market, industries and styles, from daily cross-sectional regressions | the stocks' measured exposures |

Every model is refitted each January on all earlier months (from 2005) and judged on the following year, for 2015-2024:

* **Total R²** - the share of the test months' managed-portfolio returns the model's fitted values explain, with the
  factors' realised values (how well it *describes* returns).
* **Pricing error** - the root mean square, across the 46 portfolios, of the test months' average residual: what the
  factors leave unexplained of each portfolio's *average* return (in a month's return).
* **SDF Sharpe** - each year's tangency (highest-Sharpe) combination of the factors, its weights estimated on the
  training months, held through the test year: the annualised Sharpe ratio of that out-of-sample portfolio, before
  any trading costs. A high value says the factors capture what is priced.

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display

from equity_research import charts
from equity_research.research import autoencoder as ae
from equity_research.research import labels as lb
from equity_research.research import latent as lt
from equity_research.research import linear as ln
from equity_research.research import risk_model as rm
from equity_research.research import store
from equity_research.research.cache import ResearchCache
from equity_research.research.data import load_universe_data
from equity_research.research.risk_data import load_risk_inputs
from equity_research.research.risk_styles import substyles

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
charts.style()

UNIVERSE, START, END = "SP1500", "2005-01-01", "2024-12-31"  # END: the last day before the hold-out
FIRST_TEST, LAST_TEST, KS, RP_GAMMA = 2015, 2024, (1, 3, 5), 10.0
cache = ResearchCache()
SNAPSHOT = store.data_snapshot(cache)
data = load_universe_data(cache, UNIVERSE, START, END)
dates = data.rebalances
FEATURES = {"universe": UNIVERSE, "start": START, "end": END, "dates": "month ends", "version": 1}
long = store.load("features", "company_month_end", FEATURES, snapshot=SNAPSHOT)
assert long is not None, "run notebook 6.2 first: it builds the features for this data snapshot"
monthly = store.cached(
    "labels",
    "fwd_ret_month",
    {"universe": UNIVERSE, "start": START, "end": END, "dates": "rebalances"},
    lambda: lb.period_returns(data.returns, dates),
    snapshot=SNAPSHOT,
)
x = ln.centred_ranks(long)
months = lt.monthly_panel(x, lt.clipped_returns(monthly, data.members))
managed = lt.managed_portfolios(months, list(x.columns))
print(f"{len(months)} months, {managed.shape[1]} managed portfolios, about {len(months[-1].r):,} stocks a month")

left out 887 facts of 2 companies in ['GBP']


239 months, 46 managed portfolios, about 1,503 stocks a month


## Step 1 - how many factors? The managed portfolios' variation, in sample

The share of the managed portfolios' second moment each principal component carries, over all months 2005-2024 - a
first look at how concentrated the variation is (in sample, so only a guide).

In [2]:
values = np.linalg.eigvalsh(managed.to_numpy().T @ managed.to_numpy() / len(managed))[::-1]
share = pd.Series(values / values.sum(), index=range(1, len(values) + 1))
display(share.head(8).to_frame("share of second moment").T.style.format("{:.1%}"))
print(f"The first component carries {share.iloc[0]:.0%}; the first five {share.iloc[:5].sum():.0%}")

,1,2,3,4,5,6,7,8
share of second moment,91.7%,3.8%,1.0%,0.7%,0.7%,0.4%,0.3%,0.2%


The first component carries 92%; the first five 98%


## Step 2 - the risk model's factors, month by month

Notebook 4.3's model of the S&P 1500 (market, industries, styles) gives daily factor returns from its cross-sectional
regressions; compounded over the month after each month end, they line up with the managed portfolios' months.

In [3]:
inputs = load_risk_inputs(cache, data)
exposures = rm.style_exposures(substyles(inputs), inputs)
model = rm.build_risk_model(data.returns, exposures, [pd.DatetimeIndex(data.returns.index)[-1]])
bfre = lb.period_returns(model.factor_returns, dates).reindex(managed.index).dropna(how="all")
styles = [s for s in exposures.styles if s in bfre.columns]
print(f"{bfre.shape[1]} risk-model factors over {len(bfre)} months; styles: {', '.join(styles)}")

left out 157 facts of 1 companies in ['GBP']


24 risk-model factors over 238 months; styles: Size, Volatility, Momentum, Reversal, Liquidity, Value, Earnings Yield, Dividend Yield, Profitability, Growth, Leverage, Mid-Cap


## Step 3 - every model, out of sample 2015-2024

PCA, RP-PCA, IPCA and the autoencoder with 1, 3 and 5 factors; the risk model with all its factors, with the styles
and market only, and with the market alone. The results are kept in the artifact store.

In [4]:
def everything() -> pd.DataFrame:
    """Every model's out-of-sample measures."""
    rows = {}
    for k in KS:
        rows[("PCA", k)] = lt.walk_forward_pca(managed, k, first_test=FIRST_TEST, last_test=LAST_TEST)
        rows[("RP-PCA", k)] = lt.walk_forward_pca(
            managed, k, gamma=RP_GAMMA, first_test=FIRST_TEST, last_test=LAST_TEST
        )
        rows[("IPCA", k)] = lt.walk_forward_ipca(months, k, first_test=FIRST_TEST, last_test=LAST_TEST)
        rows[("Autoencoder", k)] = ae.walk_forward(months, k, first_test=FIRST_TEST, last_test=LAST_TEST)
    for label, cols in (
        ("BFRE, all", list(bfre.columns)),
        ("BFRE, market + styles", [rm.MARKET, *styles]),
        ("BFRE, market", [rm.MARKET]),
    ):
        rows[(label, len(cols))] = lt.walk_forward_observed(
            managed, bfre[cols], first_test=FIRST_TEST, last_test=LAST_TEST
        )
    table = pd.DataFrame(rows).T
    table.index = [f"{m}, K={k}" for m, k in table.index]
    return table


table = store.cached(
    "latent",
    "oos_measures",
    FEATURES | {"ks": list(KS), "gamma": RP_GAMMA, "first_test": FIRST_TEST},
    everything,
    snapshot=SNAPSHOT,
)
display(table.style.format({"total R2": "{:.1%}", "pricing error": "{:.3%}", "SDF Sharpe": "{:+.2f}"}))

,total R2,pricing error,SDF Sharpe
"PCA, K=1",90.8%,0.046%,+0.52
"RP-PCA, K=1",90.6%,0.039%,+0.52
"IPCA, K=1",91.9%,0.050%,+0.49
"Autoencoder, K=1",91.8%,0.051%,+0.46
"PCA, K=3",95.7%,0.017%,+0.89
"RP-PCA, K=3",95.6%,0.012%,+0.92
"IPCA, K=3",97.1%,0.019%,+0.91
"Autoencoder, K=3",92.0%,0.050%,+0.51
"PCA, K=5",97.1%,0.017%,+0.79
"RP-PCA, K=5",96.8%,0.012%,+0.84


## Findings

**In short: five IPCA factors describe and price the characteristic portfolios better, out of sample, than the risk
model's twenty-four named factors.** The neural autoencoder, at the modest size used here, does not beat IPCA.

**1. One factor dominates, so read the R-squared differences, not the levels (step 1).** The first principal
component carries 91.7 % of the managed portfolios' second moment, and the first five 98 %. Among the 46 portfolios
is the equal-weighted market, whose swings dwarf those of the 45 characteristic portfolios. Every model that
captures the market therefore starts near 90 %: the market factor alone explains 85.5 % out of sample. The
informative comparisons are the remaining few percent and, above all, the pricing errors.

**2. The out-of-sample comparison, 2015-2024 (step 3):**

| Model | total R2 | pricing error (a month) | SDF Sharpe |
|---|---|---|---|
| IPCA, 5 factors | 98.2 % | 0.011 % | +1.29 |
| IPCA, 3 factors | 97.1 % | 0.019 % | +0.91 |
| PCA, 5 factors | 97.1 % | 0.017 % | +0.79 |
| RP-PCA, 5 factors | 96.8 % | 0.012 % | +0.84 |
| BFRE, market and 12 styles | 94.5 % | 0.021 % | +0.89 |
| BFRE, all 24 factors | 94.5 % | 0.020 % | +0.30 |
| Autoencoder, 5 factors | 92.0 % | 0.049 % | +1.16 |
| BFRE, market only | 85.5 % | 0.063 % | +0.72 |

* **IPCA** is best on every measure at five factors. Letting each stock's loadings follow its characteristics -
  linearly - captures more of what the characteristic portfolios do than fixed loadings (PCA) or measured exposures
  to named styles (BFRE).
* **RP-PCA** prices better than PCA with the same number of factors (pricing errors of 0.012 % against 0.017 % at
  three and five), for slightly less variance explained: as Lettau and Pelger argue, steering the factors towards
  what earns helps pricing.
* **BFRE's named factors** explain 94.5 % with 13 or 24 factors and price about as well as three latent ones. Its
  24-factor tangency portfolio is the worst SDF (Sharpe 0.30): with 24 factors, the weights estimated on past months
  are too noisy to hold up. The 13 factors of the market and styles give 0.89. The risk model was built to forecast
  risk day by day, not to price average returns - the job notebook 4.3 tests it on.
* **The conditional autoencoder** did not learn more than one factor's worth of structure: its R-squared is 92 %
  with one factor and with five. With one random start, a small network and early stopping on a year of data, it
  stopped before it could match IPCA. Gu, Kelly and Xiu average many starts and tune more, at a cost of computing this run
  avoided here, so the result says that the extra flexibility is not free, not that it cannot help.

**3. What the SDF Sharpe ratios mean.** They are the Sharpe ratios of factor portfolios rebalanced monthly across
1,500 long-short positions, with weights from past data, before any trading cost. IPCA's 1.29 says its factors
capture much of what is priced in these characteristics. It is not a tradable strategy's return: notebook 6.6
measures, for the strategies of 6.5, how much of a paper edge survives trading costs.

**4. Where this leaves the plan.** The cross-section of these 45 characteristics has a low-dimensional structure:
three to five latent factors capture almost all of it, and characteristic-driven loadings capture it best. What
the characteristics themselves predict is a separate question, answered in 6.2, 6.3 and 6.9: little beyond a
low-volatility and illiquidity effect.


## Glossary

* **Latent factor** - a driver of returns estimated from the data rather than named in advance.
* **Principal component analysis (PCA)** - the directions along which a set of series varies most.
* **Characteristic-managed portfolio** - a portfolio of all stocks weighted by one characteristic.
* **Loadings (betas)** - how much a stock or portfolio moves with each factor.
* **Pricing error (alpha)** - the part of a portfolio's average return its factor loadings do not explain.
* **Stochastic discount factor (SDF)** - in asset pricing, the combination of factors that prices every asset; its
  highest-Sharpe portfolio is the tangency portfolio of the factors.
* **Autoencoder** - a neural network that compresses its input to a few numbers and rebuilds it; *conditional*, here,
  because the loadings depend on the stocks' characteristics.